# Ionosphere — why dual-frequency GNSS works

**Goal:** understand the dispersive ionospheric delay and build the GPS L1/L2 **ionosphere-free code combination**.

Free electrons in the ionosphere affect radio signals in a frequency-dependent way. To **first order**, the extra *group delay* in a code pseudorange is

\[I(f)=\frac{40.3\;\mathrm{STEC}}{f^2}\quad[\mathrm m],\]

where `STEC` is the slant total electron content in electrons/m² and \(f\) is in Hz. **1 TECU = \(10^{16}\) electrons/m²**. The ionosphere typically *delays code* but *advances carrier phase*; the phase ionospheric correction has the opposite sign.

This exercise uses **synthetic** TEC values and noiseless pseudoranges before adding a simple noise example.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# GPS L1 and L2 frequencies. Use two different frequencies.
f1 = 1575.42e6  # Hz
f2 = 1227.60e6  # Hz
TECU = 1e16     # electrons/m²

# Exaggerated slant TEC to make the example visible.
stec_tecu = 30.0
I1 = 40.3 * stec_tecu * TECU / f1**2
I2 = 40.3 * stec_tecu * TECU / f2**2
print(f'STEC = {stec_tecu:.0f} TECU')
print(f'GPS L1 code delay: {I1:.3f} m')
print(f'GPS L2 code delay: {I2:.3f} m')
print('L2 delay is larger because its frequency is lower.')
assert I2 > I1 > 0

## 1. Derive the ionosphere-free combination

Write the code observations (ignoring noise and hardware biases for now) as

\[P_1=\rho + I_1,\qquad P_2=\rho + I_2.\]

Because \(I_i\propto 1/f_i^2\), we choose coefficients that preserve \(\rho\) and cancel the first-order ionosphere:

\[\boxed{P_{\mathrm{IF}}=\frac{f_1^2 P_1-f_2^2 P_2}{f_1^2-f_2^2}}
        =\alpha P_1+\beta P_2.\]

This is the same combination used later in code-only positioning, but **tropospheric delay, receiver/satellite clocks, orbit errors and code biases are not eliminated** by it.

In [ ]:
rho = 22_000_000.0  # toy geometric range, metres
p1 = rho + I1
p2 = rho + I2
alpha = f1**2/(f1**2-f2**2)
beta  = -f2**2/(f1**2-f2**2)
p_if = alpha*p1 + beta*p2

print(f'IF coefficients: alpha={alpha:.6f}, beta={beta:.6f}')
print(f'Uncorrected L1 error: {p1-rho:+.4f} m')
print(f'Uncorrected L2 error: {p2-rho:+.4f} m')
print(f'First-order IF error: {p_if-rho:+.8f} m')
assert np.isclose(alpha+beta, 1.0)
assert abs(p_if-rho)<1e-7

## 2. Watch the error grow with electron content

L1 and L2 code delays rise with slant TEC. In this idealized first-order model, the IF error stays approximately zero for any TEC.

In [ ]:
tecs=np.linspace(0,100,101)
ion_l1=40.3*tecs*TECU/f1**2
ion_l2=40.3*tecs*TECU/f2**2
ion_if=alpha*ion_l1+beta*ion_l2
fig,ax=plt.subplots(figsize=(7.6,4))
ax.plot(tecs,ion_l1,label='GPS L1 code error')
ax.plot(tecs,ion_l2,label='GPS L2 code error')
ax.plot(tecs,ion_if,'--',label='IF first-order error')
ax.set(xlabel='Slant TEC [TECU]',ylabel='Ionospheric code delay [m]',
       title='Dual-frequency cancellation of the first-order ionosphere')
ax.grid(alpha=.3)
ax.legend()
plt.show()
assert np.max(np.abs(ion_if)) < 1e-12

## 3. Cancellation has a price: amplified code noise

The IF combination uses a **negative coefficient**. If L1 and L2 code noises are independent with the same standard deviation \(\sigma\), then

\[\sigma_{\mathrm{IF}}=\sigma\sqrt{lpha^2+eta^2}.\]

So using both frequencies does not magically improve everything. It removes the largest frequency-dependent term but usually increases code noise and multipath sensitivity.

In [ ]:
sigma_code = 0.30  # metres, illustrative per-frequency code noise
sigma_if = sigma_code * np.sqrt(alpha**2 + beta**2)
print(f'Input code standard deviation: {sigma_code:.3f} m')
print(f'IF code standard deviation:    {sigma_if:.3f} m')
print(f'Noise amplification factor:   {sigma_if/sigma_code:.2f} times')
assert sigma_if > sigma_code

### Remember

- Code (group) and phase respond with **opposite** signs at first order.
- The IF combination cancels the **first-order** term only; higher-order terms and residual effects remain.
- It does not remove neutral-atmosphere delay, clock errors, or hardware code biases.
- Other GNSS systems/frequency pairs use their **own frequencies** (e.g. GLONASS channels).

**Question:** Why can't we eliminate the ionosphere with two pseudoranges measured at the same frequency?

Reference: [IGS products and ionospheric grids](https://igs.org/products/).